<a href="https://colab.research.google.com/github/Ronglawan/PROJECT_PHY_483/blob/main/MINI_PROJECT_PHY_483.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

#Load dataset

In [ ]:
df = pd.read_csv("ev_battery_qc_data_2026_kaggle.csv")

print(df.head())

print(df.info())

print(df.describe())

##Data preprocessing

In [ ]:
df = df.drop(['Cell_ID','Batch_ID','Inspector_Comment'],axis=1)

le = LabelEncoder()

df['Production_Line'] = le.fit_transform(df['Production_Line'])
df['Shift'] = le.fit_transform(df['Shift'])
df['Supplier'] = le.fit_transform(df['Supplier'])
df['Defect_Type'] = le.fit_transform(df['Defect_Type'])
df['QC_Grade'] = le.fit_transform(df['QC_Grade'])

##Split feature และ target

In [ ]:
X = df.drop('QC_Grade',axis=1)

y = df['QC_Grade']

##Train test split

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(
    X,y,
    test_size=0.2,
    random_state=42
)

##Normalize data

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)

##Build Deep Learning Model

In [ ]:
model = keras.Sequential([

layers.Dense(64,activation='relu'),

layers.Dense(32,activation='relu'),

layers.Dense(16,activation='relu'),

layers.Dense(3,activation='softmax')

])

##Compile model

In [ ]:
model.compile(

optimizer='adam',

loss='sparse_categorical_crossentropy',

metrics=['accuracy']

)

##Train model

In [ ]:
history = model.fit(

X_train,
y_train,

epochs=50,

batch_size=32,

validation_split=0.2

)

##Prediction

In [ ]:
y_pred = model.predict(X_test)

y_pred = np.argmax(y_pred,axis=1)

##Evaluation

In [ ]:
print("Accuracy:",accuracy_score(y_test,y_pred))

cm = confusion_matrix(y_test,y_pred)

sns.heatmap(cm,annot=True,fmt='d')

plt.xlabel("Predicted")

plt.ylabel("Actual")

plt.show()

print(classification_report(y_test,y_pred))

##Plot training graph

In [ ]:
plt.plot(history.history['accuracy'])

plt.plot(history.history['val_accuracy'])

plt.title("Model Accuracy")

plt.ylabel("Accuracy")

plt.xlabel("Epoch")

plt.legend(['Train','Validation'])

plt.show()

##Early stopping

In [ ]:
callback = keras.callbacks.EarlyStopping(
monitor='val_loss',
patience=5
)

model.fit(
X_train,
y_train,
epochs=100,
callbacks=[callback]
)

##Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier()

rf.fit(X_train,y_train)

pred = rf.predict(X_test)

print(accuracy_score(y_test,pred))